In [0]:
--Create test schema and tables
USE CATALOG workspace;
CREATE SCHEMA IF NOT EXISTS demo_sf_share
COMMENT 'Schema for Snowflake external access demo';

CREATE OR REPLACE TABLE workspace.demo_sf_share.trips_small
AS
SELECT *
FROM samples.nyctaxi.trips
LIMIT 10000;

CREATE OR REPLACE TABLE workspace.demo_sf_share.dim_zip AS
SELECT DISTINCT pickup_zip AS zip
FROM workspace.demo_sf_share.trips_small
WHERE pickup_zip IS NOT NULL

UNION

SELECT DISTINCT dropoff_zip AS zip
FROM workspace.demo_sf_share.trips_small
WHERE dropoff_zip IS NOT NULL;

CREATE OR REPLACE TABLE workspace.demo_sf_share.trips_small_iceberg
USING ICEBERG
AS
SELECT * FROM workspace.demo_sf_share.trips_small;


CREATE OR REPLACE TABLE workspace.demo_sf_share.dim_zip_iceberg
USING ICEBERG
AS
SELECT * FROM workspace.demo_sf_share.dim_zip;

--Verify tables
DESCRIBE EXTENDED workspace.demo_sf_share.trips_small_iceberg;
DESCRIBE EXTENDED workspace.demo_sf_share.dim_zip_iceberg;

--Grant EXTERNAL USE SCHEMA priviledge
GRANT EXTERNAL USE SCHEMA ON CATALOG workspace TO `304a62c1-7855-4c00-bb31-f3a16bdf3daf`;

--Grant  CATALOG priviledge
GRANT USE CATALOG ON CATALOG workspace TO `304a62c1-7855-4c00-bb31-f3a16bdf3daf`;
--Grant USE SCHEMA priviledge
GRANT USE SCHEMA ON SCHEMA workspace.demo_sf_share TO `304a62c1-7855-4c00-bb31-f3a16bdf3daf`;
--Grant SELECT on table
GRANT SELECT ON TABLE workspace.demo_sf_share.trips_small_iceberg TO `304a62c1-7855-4c00-bb31-f3a16bdf3daf`;
--Grant SELECT on table dim_zip
GRANT SELECT ON TABLE workspace.demo_sf_share.dim_zip_iceberg TO `304a62c1-7855-4c00-bb31-f3a16bdf3daf`;

--confirm grannts of Service Principals
SHOW GRANTS TO 304a62c1-7855-4c00-bb31-f3a16bdf3daf;
